# Tuần 3 — Self-RAG-inspired Pipeline (Hệ 3 trong `PIPELINE.md`)

Notebook này build **Hệ 3 — Self-RAG-inspired**, thêm 4 module phản tư mô phỏng reflection tokens của paper gốc lên trên cùng hạ tầng retrieve/generate của Notebook 1-2:

```text
Question
  -> [Retrieve] quyet dinh co can tra cuu khong
  -> Retrieve top-k dieu luat (neu can)
  -> [ISREL] danh gia tung dieu luat co lien quan khong -> loc con RELEVANT
  -> Generate answer (dua tren dieu luat da loc, hoac tra loi truc tiep neu khong co evidence)
  -> [ISSUP] answer co duoc evidence ho tro khong
  -> [ISUSE] answer co huu ich khong
  -> Answer + Reflection report
```

**Yêu cầu trước khi chạy**: đã chạy `01_retrieval_baseline.ipynb` (cần `chunks.faiss`, `chunks_meta.json`, `dev_split_qids.json`). Không phụ thuộc Notebook 2 (độc lập, tự định nghĩa lại phần Gemini client).

**Lưu ý về khối lượng gọi API**: mỗi câu hỏi giờ tốn tới ~5 lần gọi LLM thay vì 1 như Notebook 2 (retrieve-decision → ISREL → generate → ISSUP → ISUSE), nên áp lực rate-limit cao hơn hẳn. Để giảm số lần gọi, **ISREL đánh giá tất cả passage trong 1 lần gọi** (batch) thay vì gọi riêng từng passage như mô tả thuần túy trong paper — đây là đánh đổi hiệu quả có chủ đích cho điều kiện free-tier, không phải sơ suất. Nên thử với `MAX_QUESTIONS` nhỏ trước khi chạy full dev set.

**Về demo**: mục §11 (Demo) được đặt **trước** mục §12 (chạy full dev set) một cách có chủ đích — muốn demo nhanh (ví dụ lúc báo cáo cuối kỳ) chỉ cần chạy các cell từ đầu đến hết §11 (ở Colab: chuột phải vào cell demo → "Run before"), không cần chạy cell vòng lặp 250 câu ở §12 vốn tốn nhiều thời gian/API quota.

Output: `self_rag_results.jsonl` trong `ARTIFACT_DIR` — dùng làm input cho Notebook 4 (so sánh với `standard_rag_results.jsonl`).

## 0. Cấu hình môi trường + API key

Giống hệt Notebook 2: Drive chỉ chứa data/artifact (không `git clone` vào Drive), mở notebook trực tiếp từ GitHub. **API key Gemini**: lấy tại https://aistudio.google.com/apikey (Google AI Studio) — không tạo key qua Google Cloud Console/Vertex AI vì luồng đó bắt buộc bật billing (đã gặp thật trước đây). Secret Colab tên `GEMINI_API_KEY`.

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-genai

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/NLP-CS2308.CH203-data"  # sua neu ban dat ten khac
    DATA_DIR = os.path.join(DRIVE_DATA_ROOT, "VLQA")
    ARTIFACT_DIR = os.path.join(DRIVE_DATA_ROOT, "artifacts")

    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    DATA_DIR = os.path.join(REPO_DIR, "final", "dataset", "VLQA")
    ARTIFACT_DIR = os.path.join(REPO_DIR, "final", "artifacts")
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Nhap GEMINI_API_KEY: ")

print("DATA_DIR:", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR:", ARTIFACT_DIR, "| exists:", os.path.isdir(ARTIFACT_DIR))
print("GEMINI_API_KEY loaded:", bool(GEMINI_API_KEY))

## 1. Load artifact từ Notebook 1 (index, chunk metadata, dev split)

In [ ]:
import json
import faiss

INDEX_PATH = os.path.join(ARTIFACT_DIR, "chunks.faiss")
CHUNKS_META_PATH = os.path.join(ARTIFACT_DIR, "chunks_meta.json")
SPLIT_PATH = os.path.join(ARTIFACT_DIR, "dev_split_qids.json")

for path in [INDEX_PATH, CHUNKS_META_PATH, SPLIT_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Khong tim thay {path} - hay chay 01_retrieval_baseline.ipynb truoc")

index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_META_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
with open(SPLIT_PATH, encoding="utf-8") as f:
    dev_qids = set(json.load(f))

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_full = json.load(f)
dev_set = [ex for ex in train_full if ex["qid"] in dev_qids]

print(f"Da load index: {index.ntotal} chunk | dev set: {len(dev_set)} cau")

## 2. Embedding model + hàm `retrieve`

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding device:", device)


def retrieve(query, top_chunks=50, max_k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _, idxs = index.search(q_emb, top_chunks)

    results, seen = [], set()
    for idx in idxs[0]:
        c = chunks[idx]
        if c["aid"] not in seen:
            seen.add(c["aid"])
            results.append(c)
        if len(results) >= max_k:
            break
    return results

## 3. Gemini client + dò model khả dụng

Giống Notebook 2: không hardcode một model duy nhất, thử `client.models.list()` để xem tài khoản này thực sự dùng được model nào trong `CANDIDATE_MODELS` (ưu tiên `gemini-2.5-flash` → `gemini-2.0-flash` → `gemini-2.5-flash-lite` → `gemini-2.0-flash-lite`); nếu liệt kê thất bại thì dùng nguyên danh sách ưu tiên và để retry/failover ở §4 tự xử lý.

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

CANDIDATE_MODELS = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
]

try:
    available_models = {m.name.removeprefix("models/") for m in client.models.list()}
    chat_models = [m for m in CANDIDATE_MODELS if m in available_models] or CANDIDATE_MODELS
except Exception as e:
    print("Khong liet ke duoc model (dung nguyen CANDIDATE_MODELS):", e)
    chat_models = CANDIDATE_MODELS

GENERATOR_MODEL = chat_models[0]
print("Dang dung GENERATOR_MODEL =", GENERATOR_MODEL)

## 4. Hàm gọi LLM dùng chung: `chat()` (rate-limit-aware) + `extract_json()`

`chat()` là hàm **generic** (nhận prompt bất kỳ), vì 4 module reflection đều dùng chung logic gọi API này, chỉ khác prompt và cách parse output. Cơ chế rate-limit: lỗi 429 (`RESOURCE_EXHAUSTED`) có `retryDelay` ngắn thì chờ đúng thời gian đó; nếu dài (`LONG_WAIT_THRESHOLD`) thì hiểu là quota ngày của model đó đã cạn, tự động chuyển sang model tiếp theo trong `CANDIDATE_MODELS`.

Đã tắt "thinking" (`thinking_budget=0`) cho các model dòng `gemini-2.5-*` trong `gemini_config()` — tránh model tự chèn khối suy luận dài vào response ngay cả khi prompt yêu cầu chỉ trả JSON (vấn đề từng gặp thật với một model "thinking" khác khi dùng nhà cung cấp trước đây). `strip_think()` được giữ lại như một lớp phòng vệ vô hại (không có tác dụng nếu response không chứa `<think>`), áp dụng trong `extract_json()` trước khi parse và trong `generate_answer()` trước khi dùng làm câu trả lời.

`extract_json()` xử lý việc model đôi khi bọc JSON trong ```` ```json ... ``` ```` hoặc thêm chữ thừa ngoài JSON.

In [ ]:
import re
import time
from google.genai import types
from google.genai.errors import ClientError, ServerError

LONG_WAIT_THRESHOLD = 30  # giay - vuot nguong nay coi la quota ngay cua model
exhausted_models = set()


def strip_think(text):
    if not text:
        return text
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def gemini_config(model, temperature):
    kwargs = {"temperature": temperature}
    if model.startswith("gemini-2.5"):
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
    return types.GenerateContentConfig(**kwargs)


def retry_delay_seconds(exc, fallback):
    match = re.search(r'"retryDelay"\s*:\s*"(\d+(?:\.\d+)?)s"', str(exc))
    return float(match.group(1)) if match else fallback


def model_queue():
    queue = [m for m in chat_models if m not in exhausted_models]
    return queue or ([GENERATOR_MODEL] if GENERATOR_MODEL not in exhausted_models else [])


def chat(prompt, temperature=0.2, max_retries=4):
    queue = model_queue()
    if not queue:
        print("Tat ca model kha dung deu dang bi khoa dai han/loi - dung lai, thu lai sau.")
        return ""

    for model in queue:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=gemini_config(model, temperature),
                )
                return (response.text or "").strip()
            except ClientError as e:
                if e.code == 429:
                    wait = retry_delay_seconds(e, fallback=2 ** attempt)
                    if wait > LONG_WAIT_THRESHOLD:
                        print(f"Model {model} bi khoa dai han (retryDelay {wait:.0f}s) -> chuyen model tiep theo")
                        exhausted_models.add(model)
                        break
                    print(f"Rate limit model {model} (lan {attempt + 1}/{max_retries}) -> cho {wait:.1f}s")
                    time.sleep(wait)
                else:
                    print(f"Model {model} loi khong the retry (status {e.code}): {e} -> chuyen model tiep theo")
                    exhausted_models.add(model)
                    break
            except ServerError as e:
                wait = 2 ** attempt
                print(f"Loi server Gemini (lan {attempt + 1}/{max_retries}): {e} -> cho {wait}s")
                time.sleep(wait)
            except Exception as e:
                print(f"Loi khac (lan {attempt + 1}/{max_retries}): {e}")
                time.sleep(2 ** attempt)

    print("Tat ca model trong queue deu that bai (khoa dai han hoac loi) o lan chay nay.")
    return ""


def extract_json(text):
    text = strip_think(text)
    if not text:
        return None
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    match = re.search(r"(\{.*\}|\[.*\])", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return None
    return None

## 5. Module 1 — `[Retrieve]`: quyết định có cần tra cứu không

Mặc định an toàn khi parse JSON thất bại: `RETRIEVE` (thà tra cứu thừa còn hơn bỏ sót — đa số câu hỏi thật trong `train.json` đều cần retrieval).

In [ ]:
RETRIEVE_DECISION_PROMPT = """Ban la mot bo phan quyet dinh trong he thong hoi dap phap luat.
Nhiem vu: quyet dinh cau hoi sau co can tra cuu van ban luat cu the hay khong.

Tra ve RETRIEVE neu cau hoi lien quan den quy dinh, dieu luat, quyen loi, nghia vu, thu tuc phap ly cu the.
Tra ve NO_RETRIEVE neu cau hoi la giao tiep thong thuong, kien thuc pho thong khong lien quan phap luat, hoac phep toan don gian.

Cau hoi: {question}

Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"decision": "RETRIEVE hoac NO_RETRIEVE", "reason": "ly do ngan gon"}}"""


def judge_retrieve(question):
    prompt = RETRIEVE_DECISION_PROMPT.format(question=question)
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("decision") in ("RETRIEVE", "NO_RETRIEVE"):
        return result["decision"], result.get("reason", "")
    return "RETRIEVE", "Khong parse duoc JSON, mac dinh RETRIEVE de an toan"

## 6. Module 2 — `[ISREL]`: đánh giá độ liên quan (batch, 1 lần gọi cho toàn bộ passage)

Mặc định an toàn khi parse thất bại (thiếu index, JSON hỏng...): `RELEVANT` — thà giữ lại thừa một passage để generator tự cân nhắc, còn hơn mất hẳn evidence chỉ vì lỗi parse JSON.

In [ ]:
ISREL_BATCH_PROMPT = """Ban la bo phan danh gia do lien quan cua tai lieu phap luat doi voi cau hoi.

Cau hoi: {question}

Danh sach {n} dieu luat (moi dieu danh so thu tu):
{passages_block}

Voi MOI dieu luat, danh gia no co lien quan truc tiep de tra loi cau hoi khong.
Chi tra ve JSON la mot list dung {n} phan tu (moi dieu luat 1 phan tu, dung thu tu da cho), khong giai thich gi them.
Moi phan tu dang: {{"index": <so thu tu>, "label": "RELEVANT hoac IRRELEVANT", "reason": "ly do ngan gon"}}"""


def build_context(chunks_list):
    return "\n\n".join(f"[{i}] (Van ban: {c['law_id']})\n{c['text']}" for i, c in enumerate(chunks_list, start=1))


def judge_relevance_batch(question, retrieved):
    if not retrieved:
        return []
    prompt = ISREL_BATCH_PROMPT.format(question=question, n=len(retrieved), passages_block=build_context(retrieved))
    result = extract_json(chat(prompt, temperature=0))

    by_index = {}
    if isinstance(result, list):
        for item in result:
            if isinstance(item, dict) and "index" in item:
                by_index[item["index"]] = item

    labels = []
    for i, c in enumerate(retrieved, start=1):
        item = by_index.get(i)
        if item and item.get("label") in ("RELEVANT", "IRRELEVANT"):
            labels.append({"aid": c["aid"], "law_id": c["law_id"], "label": item["label"], "reason": item.get("reason", "")})
        else:
            labels.append({"aid": c["aid"], "law_id": c["law_id"], "label": "RELEVANT", "reason": "Khong parse duoc, mac dinh RELEVANT de khong mat evidence"})
    return labels

## 7. Generator (thích ứng: có evidence vs không có evidence)

Khác Standard RAG (Notebook 2, luôn có context cố định): ở đây generator dùng 2 prompt khác nhau tùy tình huống — đúng tinh thần "adaptive" của Self-RAG. Khi `[Retrieve] = NO_RETRIEVE` hoặc toàn bộ passage bị `[ISREL]` loại, hệ thống trả lời trực tiếp và **tự nói rõ không có căn cứ điều luật cụ thể** thay vì giả vờ có trích dẫn.

In [ ]:
WITH_CONTEXT_PROMPT = """Ban la tro ly tu van phap luat Viet Nam. Chi tra loi dua tren cac dieu luat duoc cung cap ben duoi. Neu cac dieu luat khong du thong tin de tra loi, hay noi ro la khong du can cu thay vi suy doan. Khi tra loi, trich dan van ban luat tuong ung bang ky hieu [so] va ghi ro ma so van ban.

Cac dieu luat lien quan:
{context}

Cau hoi: {question}

Tra loi (tieng Viet, ngan gon, co trich dan):"""

NO_CONTEXT_PROMPT = """Ban la tro ly tu van phap luat Viet Nam. Khong co dieu luat cu the nao duoc cung cap cho cau hoi nay. Hay tra loi dua tren kien thuc chung, va noi ro day khong phai trich dan tu mot dieu luat cu the.

Cau hoi: {question}

Tra loi (tieng Viet, ngan gon):"""


def generate_answer(question, relevant_chunks):
    if relevant_chunks:
        prompt = WITH_CONTEXT_PROMPT.format(context=build_context(relevant_chunks), question=question)
    else:
        prompt = NO_CONTEXT_PROMPT.format(question=question)
    return strip_think(chat(prompt, temperature=0.2))

## 8. Module 3 — `[ISSUP]`: câu trả lời có được evidence hỗ trợ không

Chỉ chạy khi có evidence (relevant_chunks không rỗng) — nếu không có evidence thì `[ISSUP]` không có ý nghĩa, orchestrator ở cell dưới sẽ gán `NOT_APPLICABLE` thay vì gọi hàm này.

In [ ]:
ISSUP_PROMPT = """Ban la bo phan kiem tra tinh duoc ho tro cua cau tra loi dua tren bang chung.

Cau hoi: {question}

Cau tra loi can kiem tra:
{answer}

Bang chung (cac dieu luat da duoc dung de tra loi):
{evidence_block}

Danh gia cau tra loi co duoc bang chung tren ho tro khong.
Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"label": "FULLY_SUPPORTED hoac PARTIALLY_SUPPORTED hoac NOT_SUPPORTED", "reason": "ly do ngan gon"}}"""


def judge_support(question, answer, relevant_chunks):
    prompt = ISSUP_PROMPT.format(question=question, answer=answer, evidence_block=build_context(relevant_chunks))
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("label") in ("FULLY_SUPPORTED", "PARTIALLY_SUPPORTED", "NOT_SUPPORTED"):
        return result["label"], result.get("reason", "")
    return "PARTIALLY_SUPPORTED", "Khong parse duoc JSON, mac dinh trung lap"

## 9. Module 4 — `[ISUSE]`: câu trả lời có hữu ích không

Luôn chạy (kể cả khi không có evidence) — usefulness đánh giá trải nghiệm người hỏi, không phụ thuộc có citation hay không.

In [ ]:
ISUSE_PROMPT = """Ban la bo phan danh gia do huu ich cua cau tra loi doi voi nguoi hoi.

Cau hoi: {question}

Cau tra loi: {answer}

Danh gia cau tra loi co huu ich, day du, va dung trong tam cau hoi khong.
Chi tra ve JSON dung dinh dang sau, khong giai thich gi them:
{{"label": "USEFUL hoac PARTIALLY_USEFUL hoac NOT_USEFUL", "reason": "ly do ngan gon"}}"""


def judge_usefulness(question, answer):
    prompt = ISUSE_PROMPT.format(question=question, answer=answer)
    result = extract_json(chat(prompt, temperature=0))
    if isinstance(result, dict) and result.get("label") in ("USEFUL", "PARTIALLY_USEFUL", "NOT_USEFUL"):
        return result["label"], result.get("reason", "")
    return "PARTIALLY_USEFUL", "Khong parse duoc JSON, mac dinh trung lap"

## 10. Orchestrator — ghép 4 module thành pipeline hoàn chỉnh

In [ ]:
def run_self_rag(question, top_chunks=50, max_k=5):
    decision, decision_reason = judge_retrieve(question)

    retrieved = []
    isrel = []
    relevant_chunks = []
    if decision == "RETRIEVE":
        retrieved = retrieve(question, top_chunks=top_chunks, max_k=max_k)
        isrel = judge_relevance_batch(question, retrieved)
        by_aid = {c["aid"]: c for c in retrieved}
        relevant_chunks = [by_aid[lbl["aid"]] for lbl in isrel if lbl["label"] == "RELEVANT"]

    answer = generate_answer(question, relevant_chunks)

    if relevant_chunks:
        issup_label, issup_reason = judge_support(question, answer, relevant_chunks)
    else:
        issup_label, issup_reason = "NOT_APPLICABLE", "Khong co evidence de doi chieu"

    isuse_label, isuse_reason = judge_usefulness(question, answer)

    return {
        "retrieve_decision": decision,
        "retrieve_decision_reason": decision_reason,
        "retrieved_aids": [c["aid"] for c in retrieved],
        "isrel": isrel,
        "relevant_aids": [c["aid"] for c in relevant_chunks],
        "generated_answer": answer,
        "issup_label": issup_label,
        "issup_reason": issup_reason,
        "isuse_label": isuse_label,
        "isuse_reason": isuse_reason,
    }

## 11. Demo — chạy thử với câu hỏi của bạn

Đặt ngay sau Orchestrator, **trước** bước chạy full dev set ở §12 — để demo nhanh (ví dụ lúc báo cáo) chỉ cần chạy các cell từ đầu notebook đến hết cell này (chuột phải → "Run before" trên cell code demo bên dưới, hoặc chạy tuần tự Shift+Enter), không phải đợi vòng lặp 250 câu ở §12 chạy xong.

Có sẵn 3 nhóm câu hỏi đúng tinh thần demo script trong guideline (§32): một câu cần retrieve, một câu không cần (kiểm tra `[Retrieve]` có tự nhận ra không), một câu "bẫy" ngoài phạm vi corpus (kiểm tra `[ISSUP]` có phát hiện thiếu căn cứ thay vì bịa không). Sửa `DEMO_QUESTIONS` và chạy lại cell bất cứ lúc nào.

In [ ]:
DEMO_QUESTIONS = [
    "Người lao động nữ mang thai có được đơn phương chấm dứt hợp đồng lao động không?",  # can retrieve
    "2 cong 2 bang may?",  # khong can retrieve
    "Self-RAG co loai bo hoan toan hallucination khong?",  # cau hoi ngoai pham vi corpus (bay)
]


def print_demo(question, max_k=5):
    result = run_self_rag(question, top_chunks=50, max_k=max_k)
    print("=" * 80)
    print("Cau hoi:", question)
    print("-" * 80)
    print("[Retrieve]:", result["retrieve_decision"], "-", result["retrieve_decision_reason"])
    if result["retrieved_aids"]:
        print(f"Da retrieve {len(result['retrieved_aids'])} dieu luat, con lai {len(result['relevant_aids'])} sau [ISREL]")
    print("\nTra loi:")
    print(result["generated_answer"])
    if result["relevant_aids"]:
        print("\nNguon trich dan (aid):", result["relevant_aids"])
    print("\n[ISSUP]:", result["issup_label"], "-", result["issup_reason"])
    print("[ISUSE]:", result["isuse_label"], "-", result["isuse_reason"])
    print("=" * 80 + "\n")


for q in DEMO_QUESTIONS:
    print_demo(q)

## 12. Chạy trên dev set (checkpoint từng câu, resume-safe)

Cùng cơ chế với Notebook 2: ghi JSONL + `flush()` ngay, tự lọc bỏ bản ghi lỗi/rỗng khi resume. `MAX_QUESTIONS` để chạy thử nhanh trước — vì mỗi câu tốn ~5 lần gọi API, nên thử với vài chục câu trước khi chạy full 250 câu. **Không cần chạy cell này để demo** — chỉ cần cho việc sinh dữ liệu đánh giá dùng ở Notebook 4.

In [ ]:
MAX_QUESTIONS = None
SLEEP_BETWEEN_CALLS = 2
RETRIEVE_K = 5
RESULTS_PATH = os.path.join(ARTIFACT_DIR, "self_rag_results.jsonl")

existing_records = []
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["generated_answer"]:
                existing_records.append(rec)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        for rec in existing_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

done_qids = {rec["qid"] for rec in existing_records}
print(f"Da co san {len(done_qids)} cau tra loi thanh cong tu lan chay truoc (da loai bo cac lan loi/rong)")

questions_to_run = dev_set if MAX_QUESTIONS is None else dev_set[:MAX_QUESTIONS]

with open(RESULTS_PATH, "a", encoding="utf-8") as f:
    for ex in questions_to_run:
        if ex["qid"] in done_qids:
            continue
        result = run_self_rag(ex["question"], top_chunks=50, max_k=RETRIEVE_K)
        record = {
            "qid": ex["qid"],
            "question": ex["question"],
            "gold_answer": ex["answer"],
            **result,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(SLEEP_BETWEEN_CALLS)

print("Hoan tat.")

## 13. Soi vài kết quả + reflection report

In [ ]:
import pandas as pd

results_df = pd.read_json(RESULTS_PATH, lines=True)
print(f"So cau da chay: {len(results_df)}")
print("\nPhan bo Retrieve decision:\n", results_df["retrieve_decision"].value_counts())
print("\nPhan bo ISSUP:\n", results_df["issup_label"].value_counts())
print("\nPhan bo ISUSE:\n", results_df["isuse_label"].value_counts())

for _, row in results_df.sample(min(3, len(results_df)), random_state=0).iterrows():
    print("\n" + "=" * 80)
    print("qid:", row["qid"])
    print("Cau hoi:", row["question"])
    print("[Retrieve]:", row["retrieve_decision"], "-", row["retrieve_decision_reason"])
    print("Retrieved aid:", row["retrieved_aids"])
    print("Relevant aid (sau ISREL):", row["relevant_aids"])
    print("Answer sinh ra:", row["generated_answer"])
    print("[ISSUP]:", row["issup_label"], "-", row["issup_reason"])
    print("[ISUSE]:", row["isuse_label"], "-", row["isuse_reason"])
    print("Gold answer:", row["gold_answer"])

## 14. Bước tiếp theo

- **Notebook 4**: load cả `standard_rag_results.jsonl` (Notebook 2) và `self_rag_results.jsonl` (notebook này), chạy thêm hệ **No-RAG** (chỉ hỏi LLM, không retrieval — tái dùng `chat()` với `NO_CONTEXT_PROMPT`), rồi lập bảng so sánh 3 hệ theo Recall@k (đã có từ Notebook 1), answer quality, support rate, usefulness rate — xuất `final_comparison_table.csv` cho báo cáo.